# 17 — CNOT & Two-Qubit Gates

Meet the one gate that lets two qubits talk to each other — and use it to build the entangled Bell state from scratch.

**Prerequisites:** `01/16_tensor_and_partial_trace`, `01/08_hadamard_and_measurement_bases`.

**What you'll be able to do**
- Apply the CNOT gate and read its truth table: it flips the target qubit when the control is $|1\rangle$.
- Build the Bell state $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$ with $H$ and CNOT, rather than writing it by hand.
- See why the result is *entangled* — not a product of two single-qubit states.

Single-qubit gates can only steer qubits independently. To create *correlation* — the resource behind entanglement, teleportation, and the quantum advantage — we need a gate that acts on two qubits at once. CNOT is the workhorse, and with one Hadamard it manufactures the most famous entangled state in physics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1
from qot_course.quantum.gates import CNOT, HADAMARD, IDENTITY, apply_gate
from qot_course.quantum.composite import tensor, bell_state, partial_trace
from qot_course.quantum.density import density_matrix

np.random.seed(42)
viz.use_course_style()

## 1. Two qubits live in a tensor product

From `01/16`, two qubits combine by the tensor product: $|00\rangle = |0\rangle \otimes |0\rangle$ is a 4-component vector. The four basis states are $|00\rangle, |01\rangle, |10\rangle, |11\rangle$, where the first slot is qubit 0 and the second is qubit 1.

In [ ]:
ket = {"00": tensor(KET_0, KET_0), "01": tensor(KET_0, KET_1),
       "10": tensor(KET_1, KET_0), "11": tensor(KET_1, KET_1)}
for name, vec in ket.items():
    print(f"|{name}> =", np.real(vec).astype(int))

**Read the output.** Each two-qubit basis state is a length-4 vector with a single $1$ — the four corners of the two-qubit space. A gate on two qubits is therefore a $4\times4$ matrix. CNOT is the most important one.

## 2. The CNOT truth table

CNOT (controlled-NOT) uses qubit 0 as the **control** and qubit 1 as the **target**: it applies $X$ to the target *only when the control is* $|1\rangle$.

$$ |00\rangle \to |00\rangle,\quad |01\rangle \to |01\rangle,\quad |10\rangle \to |11\rangle,\quad |11\rangle \to |10\rangle. $$

The control $|0\rangle$ states pass through untouched; the control $|1\rangle$ states have their target flipped.

In [ ]:
for name, vec in ket.items():
    out = apply_gate(CNOT, vec)
    out_name = [k for k, v in ket.items() if np.allclose(v, out)][0]
    print(f"CNOT|{name}> = |{out_name}>")

**Read the output.** The first two rows (control $|0\rangle$) are unchanged; the last two (control $|1\rangle$) swap $|10\rangle \leftrightarrow |11\rangle$ — the target flipped. This conditional logic is what couples the qubits. In a Qiskit circuit it is `qc.cx(0, 1)` (control 0, target 1).

## 3. Building the Bell state with H and CNOT

Here is the headline. Put the control qubit into a superposition with $H$, then apply CNOT:

$$ \text{CNOT}\,(H \otimes I)\,|00\rangle = \text{CNOT}\,\tfrac{1}{\sqrt2}(|00\rangle + |10\rangle) = \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle). $$

The superposition on the control "spreads" through CNOT, tying the two qubits together. The result is the **Bell state** that `01/18` studies — and we built it from gates, not by writing the vector down.

In [ ]:
state00 = tensor(KET_0, KET_0)
after_h = apply_gate(tensor(HADAMARD, IDENTITY), state00)   # H on the control qubit
bell = apply_gate(CNOT, after_h)

print("(H .I)|00>      =", np.round(after_h, 3), " (control in superposition)")
print("CNOT(H .I)|00>  =", np.round(bell, 3), " (= (|00>+|11>)/sqrt2)")
print("matches bell_state() ?", np.allclose(bell, bell_state()))

**Read the output.** After $H$ the state is $\tfrac{1}{\sqrt2}(|00\rangle + |10\rangle)$ — the control is uncertain but the qubits are still independent. CNOT then flips the target wherever the control is $|1\rangle$, producing $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$: now the qubits agree, always both $0$ or both $1$. It matches the course's `bell_state()` exactly.

## 4. Why this is entanglement, in one picture

A *product* state factors as $|a\rangle \otimes |b\rangle$; the Bell state does not — there is no pair of single-qubit states whose tensor product gives it. We can see the difference in the density matrices: a product state's correlations are absent, while the Bell state carries off-diagonal $|00\rangle\langle 11|$ terms linking the two outcomes.

In [ ]:
product = tensor((KET_0 + KET_1) / np.sqrt(2), KET_0)   # |+> (X) |0>: separable
labels = ["|00>", "|01>", "|10>", "|11>"]
fig = viz.plot_density_matrix(density_matrix(bell), title="Bell state: correlations off the diagonal", basis_labels=labels)
plt.show()

**Read the figure.** The Bell state's density matrix has weight not only on the diagonal $|00\rangle$ and $|11\rangle$ entries but on the off-diagonal $|00\rangle\langle 11|$ corners — a coherent link between "both 0" and "both 1". A product state has no such cross terms. Those corners are entanglement made visible, and tracing out one qubit (the partial trace of `01/16`) leaves the other in the maximally mixed $I/2$ — the signature `01/18` builds on.

## Your turn

1. **Self-inverse.** Confirm $\text{CNOT}^2 = I$ — applying it twice undoes it. Why does that make sense from the truth table?
2. **Trace it out.** Compute `partial_trace(density_matrix(bell), keep=[0], dims=[2, 2])`. You should find $I/2$ — each qubit alone looks completely random, even though together they are perfectly correlated.
3. **The other control.** Build the CNOT with qubit 1 as control and qubit 0 as target (a different $4\times4$ matrix). Apply it to $|10\rangle$ and check the result against your truth table.

## What you built

- You read the CNOT truth table: it flips the target qubit exactly when the control is $|1\rangle$.
- You built the Bell state $\tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$ from $H$ and CNOT — manufacturing entanglement with gates.
- You saw why the result is entangled: off-diagonal correlations no product state can have.

You now hold the two-qubit gate that turns independent qubits into correlated ones — the gateway to entanglement, and the last piece of the operational toolkit. Beautifully done.

## What's next

In `01/18_entanglement` we study that Bell state in depth: perfect correlations, maximally mixed parts, and what makes entanglement the defining quantum resource — now built on the gates you command.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, ch. 1.3.2, 4.3 (controlled operations), Cambridge University Press (2010).
- A. Barenco et al., "Elementary gates for quantum computation", *Physical Review A* 52, 3457 (1995). DOI:10.1103/PhysRevA.52.3457

**Previous:** `notebooks/01_QuantumFoundations/16_tensor_and_partial_trace.ipynb` · **Next:** `notebooks/01_QuantumFoundations/18_entanglement.ipynb`